# 03 · OCR — Step 15
Scanned pages (no text layer) → OCR → quality gate → page records.

| Cell | What it does |
|---|---|
| 15.0 | Setup: imports, settings, load `pages_tables.jsonl` |
| 15.1 | Functions: render page → OCR lines → drop junk lines → paragraphs |
| 15.2 | Look at ONE page (learning cell) |
| 15.3 | OCR all candidate pages once, save to `ocr_lines.jsonl` |
| 15.4 | Quality gate: thresholds, manual overrides, status per page |
| 15.5 | Threshold sweep (explore only, changes nothing) |
| 15.6 | Build OCR page records (text or "please view page N" notice) |
| 15.7 | Save images of rejected pages to check by eye |
| 15.T | Tests |

Coming next: 15.8 letters from the TOC · 15.9 small text fixes · 15.H save `pages_ocr.jsonl`

## 15.0 Setup

In [ ]:
import os, re, json, copy, time
from collections import Counter
import pandas as pd
import pdfplumber
import pytesseract
from src.paths import ROOT, PDF_PATH, INTERIM, PAGES_TABLES, PAGES_OCR, OCR_LINES
from src.text_utils import normalize_chars

pd.set_option("display.max_colwidth", 100)

# Tesseract's location depends on the machine → environment variable, with the usual Windows path as default
TESSERACT_CMD = os.environ.get("TESSERACT_CMD", r"C:\Program Files\Tesseract-OCR\tesseract.exe")
pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD
print("Tesseract version:", pytesseract.get_tesseract_version())

OCR_DPI  = 300        # image sharpness for OCR
OCR_LANG = "eng"

PAGES          = [json.loads(line) for line in PAGES_TABLES.open(encoding="utf-8")]
PAGE_BY_NO     = {p["pdf_page"]: p for p in PAGES}
PRINTED        = {p["pdf_page"]: p["printed_page"] for p in PAGES}
OCR_CANDIDATES = [p["pdf_page"] for p in PAGES if p["method"] == "needs_ocr"]     # from the data, not typed
print(f"{len(PAGES)} pages loaded · {len(OCR_CANDIDATES)} need OCR")

CHECK_DIR = INTERIM / "ocr_check"          # images saved here for checking by eye
CHECK_DIR.mkdir(exist_ok=True)

## 15.1 Functions: page → lines → drop junk → paragraphs
- `ocr_lines`: one row per line, with confidence. Words below `WORD_MIN_CONF` are removed from the line.
- Line drop rules (registry, first match wins): **no real word** (side-label letters, symbols) · **low confidence** (Hindi letterhead read as English)
- `to_paragraphs`: joins kept lines using Tesseract's own paragraph ids, re-joins `recruit-` + `ment`

In [ ]:
WORD_MIN_CONF = 40      # a word below this is dropped from its line
LINE_MIN_CONF = 60      # a line whose average is below this is dropped
REAL_WORD     = re.compile(r"[A-Za-z]{3,}")

def render_page(pdf_no, dpi=OCR_DPI):
    """PDF page → picture (PIL image). OCR can only read pictures."""
    with pdfplumber.open(PDF_PATH) as pdf:
        return pdf.pages[pdf_no - 1].to_image(resolution=dpi).original

def ocr_lines(img, lang=OCR_LANG):
    """OCR → one row per line: raw (all words), text (confident words only), line confidence."""
    d = pytesseract.image_to_data(img, lang=lang, output_type=pytesseract.Output.DATAFRAME)
    d["text"] = d["text"].fillna("").astype(str).str.strip()
    d = d[(d["conf"] >= 0) & (d["text"] != "")].copy()
    d["kept"] = d["text"].where(d["conf"] >= WORD_MIN_CONF, "")
    lines = (d.groupby(["block_num", "par_num", "line_num"], sort=True)
               .agg(conf=("conf", "mean"), top=("top", "min"), n_words=("text", "size"),
                    raw=("text", " ".join),
                    text=("kept", lambda s: " ".join(t for t in s if t)))
               .reset_index())
    lines["conf"] = lines["conf"].round(1)
    return lines

# ---- line drop rules (registry: first rule that says "drop" wins) ----
LINE_DROP_RULES = {}
def drop_rule(name):
    def deco(fn):
        LINE_DROP_RULES[name] = fn
        return fn
    return deco

@drop_rule("no real word")
def no_real_word(line):
    return not REAL_WORD.search(line["text"])

@drop_rule("low confidence")
def low_confidence(line):
    return line["conf"] < LINE_MIN_CONF

def mark_dropped(lines):
    lines = lines.copy()
    lines["dropped_by"] = [next((name for name, rule in LINE_DROP_RULES.items() if rule(row)), "")
                           for _, row in lines.iterrows()]
    return lines

def to_paragraphs(lines):
    """Kept lines → paragraphs (Tesseract's paragraph ids); re-joins words split with a hyphen."""
    kept, paras = lines[lines["dropped_by"] == ""], []
    for _, g in kept.groupby(["block_num", "par_num"], sort=True):
        text = ""
        for t in g["text"]:
            text = text[:-1] + t if re.search(r"[a-z]-$", text) else (text + " " + t).strip()
        paras.append({"text": text, "conf": round(float(g["conf"].mean()), 1), "top": int(g["top"].min())})
    return paras

def first_drop(line):
    """Name of the first rule that drops this line, or '' if it is kept (used by tests)."""
    return next((name for name, rule in LINE_DROP_RULES.items() if rule(line)), "")


## 15.2 Look at ONE page (learning cell — change `PDF_NO` to explore)

In [ ]:
PDF_NO = 112

lines = mark_dropped(ocr_lines(render_page(PDF_NO)))
print("DROPPED:")
print(lines[lines["dropped_by"] != ""][["conf", "dropped_by", "raw"]].to_string())
print("\nKEPT → PARAGRAPHS:")
for p in to_paragraphs(lines):
    print(f"[{p['conf']}] {p['text'][:140]}")

## 15.3 OCR all candidate pages once, save, reuse
First run: 3–5 minutes. Later runs load the saved file in a second.
**Delete `data/interim/ocr_lines.jsonl` after changing 15.1** (cut-offs or drop rules) to force a fresh OCR.

In [ ]:
def load_or_run_ocr(pages):
    if OCR_LINES.exists():
        saved = [json.loads(l) for l in OCR_LINES.open(encoding="utf-8")]
        print(f"loaded saved OCR for {len(saved)} pages")
        return {r["pdf_page"]: pd.DataFrame(r["lines"]) for r in saved}

    results, start = {}, time.time()
    for i, n in enumerate(pages, 1):
        results[n] = mark_dropped(ocr_lines(render_page(n)))
        print(f"\r{i}/{len(pages)} pages · {time.time() - start:.0f}s", end="")
    with OCR_LINES.open("w", encoding="utf-8") as f:
        for n, df in results.items():
            f.write(json.dumps({"pdf_page": n, "lines": json.loads(df.to_json(orient="records"))}) + "\n")
    print(f"\nsaved → {OCR_LINES.relative_to(ROOT)}")
    return results

OCR_RESULTS = load_or_run_ocr(OCR_CANDIDATES)

## 15.4 Quality gate
- `OCR_ACCEPT`: all thresholds in one place
- `OCR_OVERRIDES`: pages decided by eye (later → `config/manual_decisions.json`)
- Checks (registry, all must pass) → status (registry, first match wins): `blank` · `low_quality` · `ok`

In [ ]:
OCR_ACCEPT = {
    "blank_words":    10,     # fewer readable words → blank page
    "min_words":      30,     # enough words to be useful
    "min_conf":       80.0,   # average confidence of the kept text
    "max_junk_share": 0.5,    # share of all words thrown away
}
OCR_OVERRIDES = {             # rejected after checking by eye
    127: "low_quality",       # letter continues on 128
    128: "low_quality",
}
LOW_QUALITY_MSG = ("This scanned page is too unclear to read reliably. "
                   "Please view PDF page {pdf_page} (printed page {printed}).")

ACCEPT_CHECKS = {}
def accept_check(name):
    def deco(fn):
        ACCEPT_CHECKS[name] = fn
        return fn
    return deco

@accept_check("enough words")
def enough_words(q):    return q["kept_words"] >= OCR_ACCEPT["min_words"]

@accept_check("confident text")
def confident_text(q):  return q["kept_conf"] >= OCR_ACCEPT["min_conf"]

@accept_check("not mostly junk")
def not_mostly_junk(q): return q["junk_share"] <= OCR_ACCEPT["max_junk_share"]

STATUS_RULES = {
    "blank":       lambda q: q["kept_words"] < OCR_ACCEPT["blank_words"],
    "low_quality": lambda q: bool(q["failed_checks"]),
    "ok":          lambda q: True,
}

def page_quality(n, lines):
    kept       = lines[lines["dropped_by"] == ""]
    all_words  = int(lines["n_words"].sum())
    kept_words = int(kept["text"].str.split().str.len().sum())
    q = {"pdf_page": n, "printed": PRINTED.get(n),
         "words": all_words, "kept_words": kept_words,
         "kept_conf":  round(float(kept["conf"].mean()), 1) if len(kept) else 0.0,
         "junk_share": round(1 - kept_words / all_words, 2) if all_words else 1.0}
    q["failed_checks"] = [name for name, test in ACCEPT_CHECKS.items() if not test(q)]
    q["failed_checks"] += ["manual review"] * (n in OCR_OVERRIDES)
    q["status"] = OCR_OVERRIDES.get(n, next(s for s, rule in STATUS_RULES.items() if rule(q)))
    return q

QUALITY = pd.DataFrame([page_quality(n, df) for n, df in OCR_RESULTS.items()]).sort_values("kept_conf")
QUALITY.to_csv(ROOT / "quality" / "ocr_quality.csv", index=False)
Q_BY_PAGE = {q["pdf_page"]: q for q in QUALITY.to_dict("records")}

print(QUALITY["status"].value_counts().to_string(), "\n")
QUALITY[QUALITY["status"] != "ok"]

## 15.5 Threshold sweep (explore only — changes nothing)

In [ ]:
base = (QUALITY["kept_words"] >= OCR_ACCEPT["min_words"]) & (QUALITY["junk_share"] <= OCR_ACCEPT["max_junk_share"])
print("min_conf → pages accepted")
for c in (70, 75, 80, 85, 90):
    print(f"   {c}: {int((base & (QUALITY['kept_conf'] >= c)).sum())}")

## 15.6 Build OCR page records (text for `ok`, notice for `low_quality`)

In [ ]:
STATUS_CONTENT = {        # status → what goes into the page's lines
    "ok":          lambda lines: [{"text": normalize_chars(p["text"]), "level": "body", "font": None,
                                   "size": None, "top": p["top"], "conf": p["conf"]}
                                  for p in to_paragraphs(lines)],
    "low_quality": lambda lines: [],
    "blank":       lambda lines: [],
}
STATUS_NOTICE = {         # status → message for the chatbot
    "ok":          lambda q: None,
    "low_quality": lambda q: LOW_QUALITY_MSG.format(pdf_page=q["pdf_page"], printed=q["printed"]),
    "blank":       lambda q: None,
}

def ocr_page_record(page, lines, q):
    rec = copy.deepcopy(page)
    rec["method"]      = "ocr"
    rec["status"]      = q["status"]
    rec["ocr_quality"] = {k: q[k] for k in ("words", "kept_words", "kept_conf", "junk_share", "failed_checks")}
    rec["lines"]       = STATUS_CONTENT[q["status"]](lines)
    rec["notice"]      = STATUS_NOTICE[q["status"]](q)
    return rec

OCR_PAGES = {n: ocr_page_record(PAGE_BY_NO[n], OCR_RESULTS[n], Q_BY_PAGE[n]) for n in OCR_RESULTS}

def show_ocr(n, k=4):
    r = OCR_PAGES[n]
    print(f"\nPDF {n} (printed {r['printed_page']}) · {r['status']} · {r['ocr_quality']}")
    print("   notice:", r["notice"])
    for l in r["lines"][:k]:
        print("   ", l["text"][:110])

for n in [112, 127, 128]:
    show_ocr(n)

## 15.7 Save images of rejected pages → check by eye

In [ ]:
rejected = QUALITY.loc[QUALITY["status"] == "low_quality", "pdf_page"].tolist()
for n in rejected:
    img = render_page(n)
    img.resize((img.width // 2, img.height // 2)).save(CHECK_DIR / f"page_{n}.png")
print(f"{len(rejected)} rejected pages saved → {CHECK_DIR.relative_to(ROOT)}: {rejected}")

## 15.T Tests

In [ ]:
TESTS = []
def check(group, name, get_actual, expected, desc=None):
    try:
        actual = get_actual()
        ok = expected(actual) if callable(expected) else actual == expected
    except Exception as e:
        actual, ok = f"ERROR {type(e).__name__}: {e}", False
    TESTS.append({"group": group, "test": name, "result": "PASS" if ok else "FAIL",
                  "expected": desc if callable(expected) else expected, "actual": actual})

G = "15 ocr"
check(G, "OCR candidates", lambda: len(OCR_CANDIDATES), 56)
check(G, "every candidate OCR'd", lambda: sorted(OCR_RESULTS) == sorted(OCR_CANDIDATES), True)
check(G, "statuses are known", lambda: sorted(set(QUALITY["status"]) - set(STATUS_RULES)), [])
check(G, "PDF 112 accepted", lambda: OCR_PAGES[112]["status"], "ok")
check(G, "PDF 127 + 128 rejected by manual review",
      lambda: [(n, OCR_PAGES[n]["status"], "manual review" in OCR_PAGES[n]["ocr_quality"]["failed_checks"]) for n in (127, 128)],
      [(127, "low_quality", True), (128, "low_quality", True)])
check(G, "every rejected page has a notice",
      lambda: [n for n, r in OCR_PAGES.items() if r["status"] == "low_quality" and not r["notice"]], [])
check(G, "every accepted page has text",
      lambda: [n for n, r in OCR_PAGES.items() if r["status"] == "ok" and not r["lines"]], [])
check(G, "no text on rejected or blank pages",
      lambda: [n for n, r in OCR_PAGES.items() if r["status"] != "ok" and r["lines"]], [])
check(G, "accepted pages meet min_conf (unless overridden)",
      lambda: [n for n, r in OCR_PAGES.items() if r["status"] == "ok"
               and r["ocr_quality"]["kept_conf"] < OCR_ACCEPT["min_conf"]], [])

G = "15 unit"
check(G, "drop rule: side-label letter", lambda: first_drop({"text": "O", "conf": 90.4}), "no real word")
check(G, "drop rule: Hindi read as English", lambda: first_drop({"text": "seeeel fon", "conf": 18.1}), "low confidence")
check(G, "drop rule: body line kept", lambda: first_drop({"text": "There is a significant shortage", "conf": 95.0}), "")
check(G, "paragraph joins hyphen split",
      lambda: to_paragraphs(pd.DataFrame([
          {"block_num": 1, "par_num": 1, "line_num": 1, "conf": 95.0, "top": 10, "text": "the recruit-", "dropped_by": ""},
          {"block_num": 1, "par_num": 1, "line_num": 2, "conf": 95.0, "top": 20, "text": "ment drive.", "dropped_by": ""}]))[0]["text"],
      "the recruitment drive.")

report = pd.DataFrame(TESTS)
n_pass = int((report["result"] == "PASS").sum())
print(f"{'✅' if n_pass == len(report) else '❌'}  {n_pass}/{len(report)} tests passed")
failed = report[report["result"] == "FAIL"]
failed if len(failed) else "All tests passed — nothing to show."